# 🛒 Analyse de Sentiments des Avis Amazon par Deep Learning---## Problématique> **Comment peut-on classifier automatiquement le sentiment (positif / négatif) exprimé dans les avis clients de l'application Amazon à l'aide de techniques de Deep Learning ?**Ce notebook compare **trois architectures de Deep Learning** — ANN, RNN et LSTM — ainsi que des classifieurs classiques (Naïve Bayes, SVM, Forêt Aléatoire, etc.) pour l'analyse de sentiments.

# 1. Préparation des Données

## 1.1 Importation des bibliothèques

In [ ]:
import os
import re
import string
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, Dropout, Embedding, SimpleRNN,
                                     LSTM, Bidirectional, GlobalAveragePooling1D,
                                     Input)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

import sys
print(f"Python     : {sys.version.split()[0]}")
print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("Bibliothèques importées avec succès ✅")

## 1.2 Chargement des données

Les fichiers CSV peuvent se trouver soit dans le répertoire courant, soit dans un sous-dossier `data/`.La fonction `load_csv` gère les deux cas automatiquement.

In [ ]:
def load_csv(filename):
    """Charge un CSV depuis le répertoire courant ou le sous-dossier data/."""
    if os.path.exists(filename):
        return pd.read_csv(filename)
    elif os.path.exists(os.path.join("data", filename)):
        return pd.read_csv(os.path.join("data", filename))
    else:
        raise FileNotFoundError(f"Fichier '{filename}' introuvable.")

# Chargement du dataset nettoyé
df_clean = load_csv("reviews_clean.csv")
print(f"Nombre d'avis : {len(df_clean)}")
df_clean.head()

In [ ]:
# Informations sur le dataset
df_clean.info()

## 1.3 Labellisation des sentimentsLes avis ont été labellisés selon les règles suivantes :- **Score 1-2** → Sentiment **négatif** (-1)- **Score 4-5** → Sentiment **positif** (+1)- **Score 3** → Classé **négatif** (-1) par défaut (avis mitigé)

In [ ]:
# Chargement des données labellisées
df_labelled = load_csv("reviews_labelled.csv")
print(f"Distribution des sentiments :\n{df_labelled['sentiment'].value_counts()}")
print(f"\nTotal : {len(df_labelled)} avis")
df_labelled.head()

# 2. Exploration des Données (EDA)

## 2.1 Distribution des scores et sentiments

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des scores
score_counts = df_labelled['score'].value_counts().sort_index()
colors_score = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
axes[0].bar(score_counts.index, score_counts.values, color=colors_score, edgecolor='black')
axes[0].set_xlabel('Score (étoiles)', fontsize=12)
axes[0].set_ylabel("Nombre d'avis", fontsize=12)
axes[0].set_title('Distribution des Scores', fontsize=14, fontweight='bold')
for i, v in enumerate(score_counts.values):
    axes[0].text(score_counts.index[i], v + 15, str(v), ha='center', fontweight='bold')

# Distribution des sentiments
sent_counts = df_labelled['sentiment'].value_counts()
labels_sent = ['Négatif (-1)', 'Positif (+1)']
colors_sent = ['#e74c3c', '#2ecc71']
axes[1].pie(sent_counts.values, labels=labels_sent, colors=colors_sent,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12},
            explode=(0.05, 0.05))
axes[1].set_title('Distribution des Sentiments', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Négatifs : {sent_counts.get(-1, 0)} ({sent_counts.get(-1, 0)/len(df_labelled)*100:.1f}%)")
print(f"Positifs : {sent_counts.get(1, 0)} ({sent_counts.get(1, 0)/len(df_labelled)*100:.1f}%)")

## 2.2 Sentiment par score

In [ ]:
# Tableau croisé sentiment / score
ct = pd.crosstab(df_labelled['score'], df_labelled['sentiment'], margins=True)
ct.columns = ['Négatif (-1)', 'Positif (+1)', 'Total']
ct.index.name = 'Score'
ct

In [ ]:
# Visualisation
ct_no_margin = pd.crosstab(df_labelled['score'], df_labelled['sentiment'])
ct_no_margin.columns = ['Négatif', 'Positif']
ct_no_margin.plot(kind='bar', stacked=True, color=['#e74c3c', '#2ecc71'],
                  figsize=(10, 5), edgecolor='black')
plt.title('Distribution des Sentiments par Score', fontsize=14, fontweight='bold')
plt.xlabel('Score (étoiles)')
plt.ylabel("Nombre d'avis")
plt.legend(title='Sentiment')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2.3 Analyse de la longueur des avis

In [ ]:
df_labelled['longueur'] = df_labelled['content'].apply(lambda x: len(str(x).split()))

fig, ax = plt.subplots(figsize=(10, 5))
df_labelled[df_labelled['sentiment'] == 1]['longueur'].hist(
    bins=50, alpha=0.6, color='#2ecc71', label='Positif', ax=ax)
df_labelled[df_labelled['sentiment'] == -1]['longueur'].hist(
    bins=50, alpha=0.6, color='#e74c3c', label='Négatif', ax=ax)
ax.set_xlabel('Nombre de mots')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution de la Longueur des Avis par Sentiment', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print("Longueur moyenne (positif) :", df_labelled[df_labelled['sentiment'] == 1]['longueur'].mean().round(1), "mots")
print("Longueur moyenne (négatif) :", df_labelled[df_labelled['sentiment'] == -1]['longueur'].mean().round(1), "mots")

# 3. Pré-traitement du Texte (NLP)Les étapes de pré-traitement appliquées sont :1. **Case folding** : conversion en minuscules2. **Suppression de la ponctuation**3. **Suppression des stopwords** (mots vides)4. **Lemmatisation** (réduction des mots à leur forme de base)

## 3.1 Case Folding (mise en minuscules)

In [ ]:
df_temp = pd.DataFrame(df_labelled['content'].copy())

def case_folding(text):
    return str(text).lower()

df_temp['lower'] = df_temp['content'].apply(case_folding)
print("Exemple :")
print(f"  Avant : {df_temp['content'].iloc[0][:100]}...")
print(f"  Après : {df_temp['lower'].iloc[0][:100]}...")

## 3.2 Suppression de la ponctuation

In [ ]:
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df_temp['no_punct'] = df_temp['lower'].apply(remove_punctuation)
print("Exemple :")
print(f"  Avant : {df_temp['lower'].iloc[0][:100]}...")
print(f"  Après : {df_temp['no_punct'].iloc[0][:100]}...")

## 3.3 Suppression des Stopwords

In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join([w for w in text.split() if w not in stop_words])

df_temp['no_stop'] = df_temp['no_punct'].apply(remove_stopwords)
print("Exemple :")
print(f"  Avant : {df_temp['no_punct'].iloc[0][:100]}...")
print(f"  Après : {df_temp['no_stop'].iloc[0][:100]}...")

## 3.4 Lemmatisation

In [ ]:
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

def lemmatize_text(text):
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    lemmatized = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged]
    return ' '.join(lemmatized)

df_temp['lemmatized'] = df_temp['no_stop'].apply(lemmatize_text)
print("Exemple :")
print(f"  Avant : {df_temp['no_stop'].iloc[0][:100]}...")
print(f"  Après : {df_temp['lemmatized'].iloc[0][:100]}...")

## 3.5 Nuages de Mots (Word Clouds)

In [ ]:
from wordcloud import WordCloud

pos_text = ' '.join(df_labelled[df_labelled['sentiment'] == 1]['content'].astype(str))
neg_text = ' '.join(df_labelled[df_labelled['sentiment'] == -1]['content'].astype(str))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

wc_pos = WordCloud(width=800, height=400, background_color='white',
                   colormap='Greens', max_words=100).generate(pos_text)
axes[0].imshow(wc_pos, interpolation='bilinear')
axes[0].set_title('Avis Positifs', fontsize=16, fontweight='bold', color='green')
axes[0].axis('off')

wc_neg = WordCloud(width=800, height=400, background_color='white',
                   colormap='Reds', max_words=100).generate(neg_text)
axes[1].imshow(wc_neg, interpolation='bilinear')
axes[1].set_title('Avis Négatifs', fontsize=16, fontweight='bold', color='red')
axes[1].axis('off')

plt.suptitle('Nuages de Mots par Sentiment', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3.6 Chargement des données pré-traitées

In [ ]:
# Utilisation du fichier pré-traité existant
df_processed = load_csv("reviews_post-processing.csv")
print(f"Dataset pré-traité : {df_processed.shape[0]} lignes, {df_processed.shape[1]} colonnes")
df_processed.head()

# 4. Modélisation Classique (Baseline)Avant d'appliquer le Deep Learning, nous établissons une **ligne de base** avec des classifieurs classiques utilisant la vectorisation **TF-IDF**.

## 4.1 Vectorisation TF-IDF et séparation des données

In [ ]:
# Chargement des données pré-traitées
reviews = df_processed['content'].fillna('')
sentiments = df_processed['sentiment']

# Transformer le sentiment : -1 → 0, 1 → 1
y = sentiments.map({-1: 0, 1: 1}).values

# Vectorisation TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(reviews)

print(f"Shape TF-IDF : {X_tfidf.shape}")

# Vérification des lignes vides
row_sums = X_tfidf.sum(axis=1)
zero_sum_rows = (np.asarray(row_sums).flatten() == 0).sum()
print(f"Lignes entièrement vides : {zero_sum_rows}")

In [ ]:
# Séparation train/test (80/20)
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Taille du vocabulaire TF-IDF : {len(tfidf.vocabulary_)}")
print(f"Ensemble d'entraînement : {X_train_tfidf.shape[0]} échantillons")
print(f"Ensemble de test : {X_test_tfidf.shape[0]} échantillons")
print(f"Distribution train : {np.bincount(y_train)}")
print(f"Distribution test  : {np.bincount(y_test)}")

## 4.2 Entraînement des classifieurs classiques

In [ ]:
from sklearn.naive_bayes import BernoulliNB, MultinomialNB, GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

classifiers = {
    "Bernoulli NB": BernoulliNB(),
    "Multinomial NB": MultinomialNB(),
    "Gaussian NB": GaussianNB(),
    "SVM (RBF)": SVC(kernel='rbf', random_state=42),
    "Arbre de Décision": DecisionTreeClassifier(random_state=42),
    "Forêt Aléatoire": RandomForestClassifier(n_estimators=100, random_state=42),
    "Régression Logistique": LogisticRegression(max_iter=1000, random_state=42),
}

results_ml = {}

for name, clf in classifiers.items():
    # GaussianNB nécessite une matrice dense
    if name == "Gaussian NB":
        clf.fit(X_train_tfidf.toarray(), y_train)
        y_pred = clf.predict(X_test_tfidf.toarray())
    else:
        clf.fit(X_train_tfidf, y_train)
        y_pred = clf.predict(X_test_tfidf)
    
    results_ml[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'y_pred': y_pred
    }
    print(f"✅ {name:25s} → Accuracy: {results_ml[name]['accuracy']:.4f} | F1: {results_ml[name]['f1']:.4f}")

## 4.3 Tableau récapitulatif (Baseline)

In [ ]:
df_baseline = pd.DataFrame({
    'Classifieur': list(results_ml.keys()),
    'Accuracy': [v['accuracy'] for v in results_ml.values()],
    'Precision': [v['precision'] for v in results_ml.values()],
    'Recall': [v['recall'] for v in results_ml.values()],
    'F1-Score': [v['f1'] for v in results_ml.values()],
}).sort_values('Accuracy', ascending=False).reset_index(drop=True)

df_baseline.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}'
}).background_gradient(cmap='Greens', subset=['Accuracy', 'F1-Score'])

In [ ]:
# Matrices de confusion pour les 3 meilleurs classifieurs
top3 = df_baseline['Classifieur'].head(3).tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3):
    cm = confusion_matrix(y_test, results_ml[name]['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Négatif', 'Positif']).plot(ax=axes[i], cmap='Blues')
    axes[i].set_title(name, fontsize=13, fontweight='bold')
plt.suptitle('Matrices de Confusion – Top 3 Classifieurs Classiques', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# 5. Modélisation Deep LearningNous allons maintenant construire et comparer **trois architectures de Deep Learning** pour la classification de sentiments :1. **ANN** – Réseau de neurones dense (ne capture pas l'ordre des mots)2. **RNN** – Réseau récurrent (capture les dépendances séquentielles)3. **LSTM** – Mémoire à long terme bidirectionnelle

## 5.1 Tokenisation et Padding

In [ ]:
# Paramètres
VOCAB_SIZE = 10000    # Taille du vocabulaire
MAX_LEN = 200         # Longueur maximale des séquences
EMBEDDING_DIM = 128   # Dimension des embeddings
BATCH_SIZE = 32
EPOCHS = 20

# Reproductibilité
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# Textes bruts pré-traités
texts = df_processed['content'].fillna('').values
labels = df_processed['sentiment'].map({-1: 0, 1: 1}).values

# Tokenisation avec Keras
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

# Padding (ajout de zéros pour uniformiser la longueur)
X_padded = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

print(f"Taille du vocabulaire : {min(len(tokenizer.word_index) + 1, VOCAB_SIZE)}")
print(f"Forme des séquences paddées : {X_padded.shape}")
print(f"Exemple de séquence (premiers 20 tokens) : {X_padded[0][:20]}")

In [ ]:
# Séparation train / test
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(
    X_padded, labels, test_size=0.2, random_state=SEED, stratify=labels
)

print(f"Train : {X_train_dl.shape[0]} échantillons")
print(f"Test  : {X_test_dl.shape[0]} échantillons")
print(f"Distribution train : Négatif={sum(y_train_dl == 0)}, Positif={sum(y_train_dl == 1)}")
print(f"Distribution test  : Négatif={sum(y_test_dl == 0)}, Positif={sum(y_test_dl == 1)}")

In [ ]:
# Callback pour l'arrêt anticipé (éviter le sur-apprentissage)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

## 5.2 Modèle 1 : ANN (Artificial Neural Network)Le **réseau de neurones artificiels** (ou perceptron multicouche) est un modèle dense qui ne tient pas compte de l'ordre des mots. Il utilise un `GlobalAveragePooling` sur les embeddings pour créer une représentation fixe du texte.

In [ ]:
# Construction du modèle ANN
model_ann = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),
    GlobalAveragePooling1D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
], name='ANN')

model_ann.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_ann.summary()

In [ ]:
# Entraînement du modèle ANN
print("=" * 60)
print("ENTRAÎNEMENT DU MODÈLE ANN")
print("=" * 60)

history_ann = model_ann.fit(
    X_train_dl, y_train_dl,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Évaluation ANN
loss_ann, acc_ann = model_ann.evaluate(X_test_dl, y_test_dl, verbose=0)
y_pred_ann = (model_ann.predict(X_test_dl, verbose=0) > 0.5).astype(int).flatten()

print(f"\n{'=' * 50}")
print("RÉSULTATS ANN")
print(f"{'=' * 50}")
print(f"Loss     : {loss_ann:.4f}")
print(f"Accuracy : {acc_ann:.4f}")
print(f"\nRapport de classification :")
print(classification_report(y_test_dl, y_pred_ann, target_names=['Négatif', 'Positif']))

## 5.3 Modèle 2 : RNN (Recurrent Neural Network)Le **réseau de neurones récurrent** traite les séquences de mots **pas à pas**, ce qui lui permet de capturer les dépendances entre les mots successifs.

In [ ]:
# Construction du modèle RNN
model_rnn = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),
    SimpleRNN(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
], name='RNN')

model_rnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_rnn.summary()

In [ ]:
# Entraînement du modèle RNN
print("=" * 60)
print("ENTRAÎNEMENT DU MODÈLE RNN")
print("=" * 60)

history_rnn = model_rnn.fit(
    X_train_dl, y_train_dl,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Évaluation RNN
loss_rnn, acc_rnn = model_rnn.evaluate(X_test_dl, y_test_dl, verbose=0)
y_pred_rnn = (model_rnn.predict(X_test_dl, verbose=0) > 0.5).astype(int).flatten()

print(f"\n{'=' * 50}")
print("RÉSULTATS RNN")
print(f"{'=' * 50}")
print(f"Loss     : {loss_rnn:.4f}")
print(f"Accuracy : {acc_rnn:.4f}")
print(f"\nRapport de classification :")
print(classification_report(y_test_dl, y_pred_rnn, target_names=['Négatif', 'Positif']))

## 5.4 Modèle 3 : LSTM (Long Short-Term Memory)Le **LSTM** est une variante avancée du RNN qui résout le problème du vanishing gradient grâce à des cellules mémoire spécialisées. La version **bidirectionnelle** lit la séquence dans les deux sens pour une meilleure compréhension du contexte.

In [ ]:
# Construction du modèle LSTM (Bidirectionnel)
model_lstm = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
], name='LSTM_Bidirectional')

model_lstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_lstm.summary()

In [ ]:
# Entraînement du modèle LSTM
print("=" * 60)
print("ENTRAÎNEMENT DU MODÈLE LSTM (Bidirectionnel)")
print("=" * 60)

history_lstm = model_lstm.fit(
    X_train_dl, y_train_dl,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Évaluation LSTM
loss_lstm, acc_lstm = model_lstm.evaluate(X_test_dl, y_test_dl, verbose=0)
y_pred_lstm = (model_lstm.predict(X_test_dl, verbose=0) > 0.5).astype(int).flatten()

print(f"\n{'=' * 50}")
print("RÉSULTATS LSTM (Bidirectionnel)")
print(f"{'=' * 50}")
print(f"Loss     : {loss_lstm:.4f}")
print(f"Accuracy : {acc_lstm:.4f}")
print(f"\nRapport de classification :")
print(classification_report(y_test_dl, y_pred_lstm, target_names=['Négatif', 'Positif']))

# 6. Évaluation et Comparaison des ModèlesDans cette section, nous comparons les performances de tous les modèles (classiques et Deep Learning) à l'aide de métriques standard et de visualisations.

## 6.1 Collecte des résultats Deep Learning

In [ ]:
# Dictionnaire des prédictions Deep Learning
dl_results = {
    'ANN': y_pred_ann,
    'RNN': y_pred_rnn,
    'LSTM': y_pred_lstm,
}

# Calcul des métriques DL
dl_metrics = {}
for name, y_pred in dl_results.items():
    y_pred_flat = np.asarray(y_pred).ravel()
    dl_metrics[name] = {
        'Accuracy': accuracy_score(y_test_dl, y_pred_flat),
        'Precision': precision_score(y_test_dl, y_pred_flat, zero_division=0),
        'Recall': recall_score(y_test_dl, y_pred_flat, zero_division=0),
        'F1-Score': f1_score(y_test_dl, y_pred_flat, zero_division=0),
    }

for name, m in dl_metrics.items():
    print(f"✅ {name:5s} → Accuracy: {m['Accuracy']:.4f} | F1: {m['F1-Score']:.4f}")

## 6.2 Courbes d'apprentissage (Deep Learning)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

histories = {'ANN': history_ann, 'RNN': history_rnn, 'LSTM': history_lstm}
colors = {'ANN': '#3498db', 'RNN': '#e67e22', 'LSTM': '#9b59b6'}

for i, (name, history) in enumerate(histories.items()):
    # Accuracy
    axes[0][i].plot(history.history['accuracy'], label='Train', color=colors[name], linewidth=2)
    axes[0][i].plot(history.history['val_accuracy'], label='Validation',
                    color=colors[name], linewidth=2, linestyle='--')
    axes[0][i].set_title(f'{name} – Accuracy', fontsize=13, fontweight='bold')
    axes[0][i].set_xlabel('Époque')
    axes[0][i].set_ylabel('Accuracy')
    axes[0][i].legend()
    axes[0][i].grid(True, alpha=0.3)
    
    # Loss
    axes[1][i].plot(history.history['loss'], label='Train', color=colors[name], linewidth=2)
    axes[1][i].plot(history.history['val_loss'], label='Validation',
                    color=colors[name], linewidth=2, linestyle='--')
    axes[1][i].set_title(f'{name} – Loss', fontsize=13, fontweight='bold')
    axes[1][i].set_xlabel('Époque')
    axes[1][i].set_ylabel('Loss')
    axes[1][i].legend()
    axes[1][i].grid(True, alpha=0.3)

plt.suptitle("Courbes d'Apprentissage des Modèles Deep Learning", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 6.3 Tableau Comparatif Global

In [ ]:
# Combinaison ML classique + Deep Learning
rows = []
for name, metrics in results_ml.items():
    rows.append({
        'Modèle': name,
        'Type': 'ML Classique',
        'Accuracy': metrics['accuracy'],
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1-Score': metrics['f1'],
    })
for name, metrics in dl_metrics.items():
    rows.append({
        'Modèle': name,
        'Type': 'Deep Learning',
        'Accuracy': metrics['Accuracy'],
        'Precision': metrics['Precision'],
        'Recall': metrics['Recall'],
        'F1-Score': metrics['F1-Score'],
    })

df_comparison = pd.DataFrame(rows).sort_values('Accuracy', ascending=False).reset_index(drop=True)

df_comparison.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}'
}).background_gradient(cmap='YlGn', subset=['Accuracy', 'F1-Score'])

## 6.4 Visualisation Comparative

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Graphique 1 : Accuracy
df_sorted = df_comparison.sort_values('Accuracy', ascending=True)
colors_bar = ['#3498db' if t == 'Deep Learning' else '#95a5a6' for t in df_sorted['Type']]
axes[0].barh(df_sorted['Modèle'], df_sorted['Accuracy'], color=colors_bar, edgecolor='black')
axes[0].set_xlabel('Accuracy', fontsize=12)
axes[0].set_title('Comparaison des Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlim(0.5, 1.0)
for i, v in enumerate(df_sorted['Accuracy']):
    axes[0].text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=10)

# Graphique 2 : F1-Score
df_sorted_f1 = df_comparison.sort_values('F1-Score', ascending=True)
colors_bar_f1 = ['#e74c3c' if t == 'Deep Learning' else '#95a5a6' for t in df_sorted_f1['Type']]
axes[1].barh(df_sorted_f1['Modèle'], df_sorted_f1['F1-Score'], color=colors_bar_f1, edgecolor='black')
axes[1].set_xlabel('F1-Score', fontsize=12)
axes[1].set_title('Comparaison des F1-Scores', fontsize=14, fontweight='bold')
axes[1].set_xlim(0.5, 1.0)
for i, v in enumerate(df_sorted_f1['F1-Score']):
    axes[1].text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=10)

plt.suptitle('ML Classique (gris) vs Deep Learning (couleur)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Comparaison détaillée des 3 modèles Deep Learning
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
dl_names = ['ANN', 'RNN', 'LSTM']
dl_colors = ['#3498db', '#e67e22', '#9b59b6']

x = np.arange(len(metrics_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, name in enumerate(dl_names):
    values = [dl_metrics[name][m] for m in metrics_names]
    bars = ax.bar(x + i * width, values, width, label=name, color=dl_colors[i], edgecolor='black')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparaison Détaillée des Modèles Deep Learning', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_names, fontsize=12)
ax.legend(fontsize=12)
ax.set_ylim(0.5, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 7. Test sur de Nouveaux Exemples

In [ ]:
def predict_sentiment(text, model, tokenizer, max_len=MAX_LEN):
    """Prédit le sentiment d'un texte donné."""
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    prob = model.predict(padded, verbose=0)[0][0]
    sentiment = "Positif ✅" if prob > 0.5 else "Négatif ❌"
    return sentiment, prob

# Exemples de test
examples = [
    "This app is amazing! Fast delivery and great prices. Love it!",
    "Terrible experience. The app keeps crashing and customer service is useless.",
    "It's okay, nothing special. Works most of the time.",
    "Best shopping app ever! Easy to use and reliable.",
    "Waste of time. Orders always arrive late and damaged.",
]

print("=" * 80)
print("PRÉDICTIONS SUR DE NOUVEAUX EXEMPLES")
print("=" * 80)

for text in examples:
    display_text = f'"{text[:80]}..."' if len(text) > 80 else f'"{text}"'
    print(f"\nTexte : {display_text}")
    for model, name in [(model_ann, 'ANN'), (model_rnn, 'RNN'), (model_lstm, 'LSTM')]:
        sentiment, prob = predict_sentiment(text, model, tokenizer)
        print(f"  {name:5s} → {sentiment} (probabilité: {prob:.4f})")

# 8. Conclusion## Synthèse des résultatsCe projet a permis de comparer différentes approches pour l'analyse de sentiments sur des avis Amazon :**ML Classique (Baseline)** : Les classifieurs traditionnels avec TF-IDF offrent déjà de bonnes performances, en particulier la Régression Logistique et le SVM.**Deep Learning** : Les trois architectures (ANN, RNN, LSTM) apportent des perspectives différentes :- **ANN** : Simple et rapide, mais ne capture pas l'ordre des mots- **RNN** : Capture les dépendances séquentielles, mais peut souffrir du vanishing gradient- **LSTM Bidirectionnel** : Architecture la plus sophistiquée, avec mémoire à long terme et lecture dans les deux sens## Points clés- Le dataset est déséquilibré (plus de négatifs que de positifs)- Le pré-traitement NLP (lemmatisation, suppression des stopwords) est crucial- Les modèles DL bénéficient de la capture de la séquentialité des mots- L'EarlyStopping permet d'éviter le sur-apprentissage